# NB03 · ¿Qué texto codifico? — plantillas y representación

**El montaje de NB02, al revés.** Allí el texto estaba congelado en A0 y variaba el modelo; aquí el modelo queda congelado en el ganador de R02 y **lo único que cambia es el texto** (Regla 1).

> 🔒 **Congelado** (`config.yaml` → `nb03_representacion.modelo_congelado`): `gemini-embedding-2` · contrato `sin_contrato` · **dim 768** · métrica `cosine` · normalización L2 explícita al truncar · k=10.

### La pregunta

Un embedding resume el significado de **todo** lo que entra. Si de los ~1.300 caracteres que tiene `text` de media, la mayoría es prosa comercial —*"perfecto para regalo, calidad premium, ideal para toda ocasión"*—, el vector se acerca al lenguaje genérico de cualquier producto y se aleja de lo que hace distinto a *este*.

**Menos texto puede recuperar mejor.** Es una hipótesis, y aquí se mide.

### Qué NO entra: la familia C (D07)

El chunking queda descartado **por medición, no por falta de tiempo**. NB02·A midió con el tokenizador de cada modelo que `pct_supera_ventana = 0` sobre los 15.000 registros: el máximo son 1.972 tokens frente a ventanas de 8.192 y 32.768, más de 4× de margen. Partir en trozos resuelve un problema que aquí no existe.

Consecuencia, ya registrada en `config.yaml`: el punto de la base vectorial sigue siendo `record_id` en relación **1:1** con el producto. El esquema de NB04 se mantiene simple, la idempotencia no necesita borrar chunks huérfanos, el top-10 no necesita deduplicar y **D08 queda sin aplicar**.

In [ ]:
import gc
import json
import time
from pathlib import Path

import pandas as pd

import sys
sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv
import os

from aurum.busqueda import DenseRetriever, rank_queries_dense
from aurum.datos import relevant_field_nullity
from aurum.embeddings import GeminiEncoder, encode_corpus, truncate_dim, vector_health
from aurum.evaluacion import (
    apply_tolerance_rule,
    evaluate_rankings,
    formulation_consistency,
    per_query_delta,
    qrels_from_judgements,
)
from aurum.graficas import plot_effect_vs_exposure, plot_metric_comparison
from aurum.plantillas import (
    CONTROLES,
    TEMPLATES,
    candidatas,
    corpus_context,
    render_template,
    template_stats,
)

load_dotenv(Path("..") / ".env")

DATA = Path("..") / "data"
CACHE = Path("..") / "artifacts" / "embeddings"

# Espejo de config.yaml -> nb03_representacion.modelo_congelado
MODELO, CONTRATO, DIM, METRICA = "gemini-embedding-2", "sin_contrato", 768, "cosine"
CORPUS_ID = "catalogo_muestra"      # condición 3 del plan
TOP_K = 10
TOLERANCIA_R01 = 0.01               # r01_criterio_desempate.tolerancia

muestra = pd.read_csv(DATA / "catalogo_muestra.csv")
consultas = pd.read_csv(DATA / "consultas_desarrollo.csv")
ciegas = pd.read_csv(DATA / "consultas_evaluacion.csv")
relevancias = pd.read_csv(DATA / "relevancias_desarrollo.csv")
qrels = qrels_from_judgements(relevancias)

corpus_ids = muestra["product_id"].tolist()
query_ids = [str(q) for q in consultas["query_id"]]
query_textos = consultas["query_text"].tolist()

CONTEXTO = corpus_context(muestra)

print(f"catálogo : {len(muestra)} productos")
print(f"corte A4 : {CONTEXTO.a4_chars} caracteres (derivado del corpus, ver A.2)")
print(f"consultas: {len(query_textos)} de desarrollo · {len(ciegas)} ciegas")
print(f"congelado: {MODELO} [{CONTRATO}] @{DIM} · {METRICA}")

catálogo : 1500 productos
corte A4 : 1083 caracteres (derivado del corpus, ver A.2)
consultas: 8 de desarrollo · 12 ciegas
congelado: gemini-embedding-2 [sin_contrato] @768 · cosine


## A · Las siete plantillas (D06)

Cada plantilla es una receta para construir la cadena que se codifica. Viven en `aurum.plantillas` y no en el notebook porque son **el objeto de estudio**: tienen que poder probarse sin red y sin modelo (`tests/test_plantillas.py`).

| | Texto | Por qué está |
|---|---|---|
| **A0** | `text` tal cual | La que NB02 tuvo congelada. Sin ella no hay punto de comparación |
| **A1** | solo `title` | El extremo opuesto, y la única que nunca podría truncarse |
| **A2** | `title` + marca + color, sin etiquetas | Aísla si lo que aporta A3 es la información o la nomenclatura |
| **A3** | con etiquetas (`Marca: X. Color: Y.`), omitiendo vacíos | Replica la nomenclatura del `text` de origen, aplicando **D02** |
| **A3n** | igual que A3 pero rellenando los vacíos | **El control de D02** — ver abajo |
| **A4** | recorte de `text` por la **mediana del corpus**, en frontera de palabra | El único punto intermedio entre A0 y A3. El corte no lo elige nadie: sale de los datos (A.2) |
| **A5** | A3 sin `color` | `color` falta en el 36,6 %: ¿aporta o estorba? |

### 🔬 Por qué A3n no se llama A6

Porque no es otra receta de la secuencia: es **el control de A3**. D02 decidió omitir la sección de un campo vacío en lugar de escribir `"Color: desconocido"`, y el argumento fue que insertar un literal compartido en el 36,6 % del catálogo crearía una señal común artificial — productos que se acercan por compartir una palabra, no por parecerse.

Ese razonamiento era sólido pero **no estaba medido**, y §3.1 no da por buena una justificación sin datos. A3 frente a A3n aísla exactamente esa política: si contamina, A3n saldrá peor; si da igual, D02 era una precaución sin coste; y si sale mejor, la decisión estaba equivocada y se descubre a tiempo.

In [ ]:
template_stats(muestra)

### A.1 · El mismo producto por las siete recetas

Ver el texto real es lo que evita discutir sobre abstracciones. Fíjate en la distancia entre A0 y el resto: si A3 gana, la conclusión no será *"las etiquetas ayudan"* sino que **el resto del texto era relleno**.

In [ ]:
fila = muestra.head(1)
for nombre in TEMPLATES:
    texto = render_template(fila, nombre, context=CONTEXTO)[0]
    print(f"\n──── {nombre}  ({len(texto)} chars) " + "─" * 40)
    print(texto[:300] + ("…" if len(texto) > 300 else ""))


──── A0  (3000 chars) ────────────────────────────────────────
Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Talla Grande Elegante De Manga Larga con Escote En La Manga De Navidad, Vestido Largo De Noche De Fiesta De Noche De Playa. Marca: KanLin1986-Ropa. Color: Negro. Características: 🔥🔥Ropa Vestidos para niña Vestidos para mujer Vestidos Ropa de da…

──── A1  (178 chars) ────────────────────────────────────────
Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Talla Grande Elegante De Manga Larga con Escote En La Manga De Navidad, Vestido Largo De Noche De Fiesta De Noche De Playa

──── A2  (200 chars) ────────────────────────────────────────
Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Talla Grande Elegante De Manga Larga con Escote En La Manga De Navidad, Vestido Largo De Noche De Fiesta De Noche De Playa KanLin1986-Ropa Negro

──── A3  (216 chars) ────────────────────────────────────────
Kanlin1986 Vestido Largo De Navidad para Mujer, Vestido Talla Gra

### A.2 · De dónde sale el recorte de A4

A4 recorta `text`, pero **el punto de corte no lo elige nadie**: se deriva del propio corpus. Un número escrito a mano —512, pongamos— sería una decisión de diseño disfrazada de detalle de implementación, imposible de justificar frente a 400 o 600 y sin sentido en cuanto cambiara el catálogo.

Se usa la **mediana** de `text`, no la media: la distribución está sesgada a la derecha y topada en 3.000 caracteres, así que la media queda por encima de lo típico y apenas tocaría a cuatro de cada diez fichas. La mediana parte el catálogo en dos mitades exactas y convierte A4 en una pregunta nítida: **¿sobra la mitad más larga de cada ficha?**

In [ ]:
longitudes = muestra["text"].fillna("").str.len()

pd.DataFrame([{
    "corte_A4": CONTEXTO.a4_chars,
    "origen": "mediana de `text`",
    "media": round(float(longitudes.mean()), 1),
    "mediana": int(longitudes.median()),
    "p90": int(longitudes.quantile(0.9)),
    "max": int(longitudes.max()),
    "pct_productos_recortados": round(100 * float((longitudes > CONTEXTO.a4_chars).mean()), 1),
}])

,corte_A4,origen,media,mediana,p90,max,pct_productos_recortados
0,1083,mediana de `text`,1308.9,1083,2999,3000,50.0


### A.3 · Ninguna plantilla se trunca

No hace falta volver a medirlo: NB02 comprobó que **A0 —la más larga de las siete— no supera la ventana en ningún registro** del catálogo completo (máximo 1.972 tokens frente a 8.192). Todas las demás son estrictamente más cortas que A0, así que ninguna puede truncarse.

Esto vacía de contenido la familia B del plan como *medición de truncado*, pero no la pregunta que había detrás. La reformulamos: ya no es *"¿se pierde información al truncar?"* sino **"¿el texto largo es señal o es relleno?"** — y esa la responde A4 frente a A0.

## B · Codificar las siete variantes

⏱️ **~6 minutos.** Gemini codifica 1.500 documentos en ~50 s, así que las siete salen por unos 6 min. Con los modelos locales de NB02 (5 h y 14 h) este barrido habría sido inviable — es una consecuencia directa de qué modelo ganó R02.

⚠️ **Cada plantilla invalida la caché**, y eso es lo correcto: `encode_corpus` incluye el SHA-256 del corpus en el nombre del artefacto, así que cambiar el texto cambia la huella y fuerza a recodificar. Es justo lo que impide comparar en silencio vectores de dos textos distintos.

**Las consultas no se recodifican.** Las plantillas describen *productos*; una consulta es lo que escribe la persona y no tiene plantilla que aplicar. Sus vectores ya están en `artifacts/embeddings/` desde NB02.

In [ ]:
VECTORES_PLANTILLA = {}
COSTES_PLANTILLA = {}
ERRORES_PLANTILLA = {}


def encoder_congelado():
    """El ganador de R02. Se construye por llamada y se libera después."""
    return GeminiEncoder(
        api_key=os.environ.get("GEMINI_API_KEY"),
        model_id=MODELO,
        native_dim=3072,
        window=8192,
    )


def codificar_plantilla(nombre):
    """Codifica el catálogo con una plantilla y deja el resultado en memoria."""
    textos = render_template(muestra, nombre)
    encoder = encoder_congelado()
    try:
        resultado = encode_corpus(
            encoder, textos, corpus_id=f"{CORPUS_ID}__{nombre}",
            kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
        )
    finally:
        del encoder
        gc.collect()
    VECTORES_PLANTILLA[nombre] = resultado.vectors
    COSTES_PLANTILLA[nombre] = {"plantilla": nombre, **resultado.stats.as_row()}
    return resultado


for nombre in TEMPLATES:
    inicio = time.perf_counter()
    try:
        r = codificar_plantilla(nombre)
        origen = "caché" if r.stats.desde_cache else f"{time.perf_counter() - inicio:.1f}s"
        print(f"✅ {nombre:<4} {r.vectors.shape} ({origen})")
    except Exception as error:
        ERRORES_PLANTILLA[nombre] = f"{type(error).__name__}: {error}"
        print(f"⛔ {nombre:<4} {ERRORES_PLANTILLA[nombre][:120]}")

✅ A0   (1500, 3072) (caché)
✅ A1   (1500, 3072) (caché)
✅ A2   (1500, 3072) (caché)
✅ A3   (1500, 3072) (caché)
✅ A3n  (1500, 3072) (caché)
✅ A4   (1500, 3072) (caché)
✅ A5   (1500, 3072) (caché)


### B.1 · Consultas y salud de los vectores

Las consultas se leen de la caché de NB02 — mismo modelo, mismo contrato, mismo corpus de consultas. Y antes de creerse ninguna métrica, las comprobaciones de siempre: sin `NaN`, normas coherentes y sin filas duplicadas.

Un dato a vigilar en la tabla: **`n_filas_duplicadas`**. Si una plantilla produce vectores idénticos para productos distintos, es que está tirando la información que los diferencia — y eso se ve aquí antes que en el nDCG.

In [ ]:
encoder = encoder_congelado()
try:
    vectores_query = encode_corpus(
        encoder, query_textos, corpus_id="consultas_desarrollo",
        kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
    ).vectors
    vectores_ciegas = encode_corpus(
        encoder, ciegas["query_text"].tolist(), corpus_id="consultas_evaluacion",
        kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
    ).vectors
finally:
    del encoder
    gc.collect()

print(f"consultas de desarrollo: {vectores_query.shape}")
print(f"consultas ciegas       : {vectores_ciegas.shape}")

pd.DataFrame([
    {"plantilla": nombre, **vector_health(truncate_dim(matriz, DIM))}
    for nombre, matriz in VECTORES_PLANTILLA.items()
])

consultas de desarrollo: (8, 3072)
consultas ciegas       : (12, 3072)


,plantilla,n_vectores,dim,dtype,finito,norma_min,norma_max,normalizado,n_filas_duplicadas,bytes_por_vector
0,A0,1500,768,float32,True,1.0,1.0,True,0,3072
1,A1,1500,768,float32,True,1.0,1.0,True,0,3072
2,A2,1500,768,float32,True,1.0,1.0,True,0,3072
3,A3,1500,768,float32,True,1.0,1.0,True,0,3072
4,A3n,1500,768,float32,True,1.0,1.0,True,0,3072
5,A4,1500,768,float32,True,1.0,1.0,True,0,3072
6,A5,1500,768,float32,True,1.0,1.0,True,0,3072


## C · El barrido (R01)

Cada plantilla se evalúa con el modelo congelado a 768 dimensiones sobre las mismas 8 consultas, los mismos qrels y el mismo `k`. Solo cambia el texto de los documentos.

In [ ]:
def evaluar_plantilla(nombre, *, dim=DIM, metric=METRICA):
    """Evalúa una plantilla sobre las 8 consultas de desarrollo."""
    docs = truncate_dim(VECTORES_PLANTILLA[nombre], dim)
    queries = truncate_dim(vectores_query, dim)
    retriever = DenseRetriever(docs, corpus_ids, metric=metric)
    rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
    return evaluate_rankings(rankings, qrels, k=TOP_K), rankings


longitudes = template_stats(muestra).set_index("plantilla")["chars_media"]

BARRIDO_PLANTILLAS = []
RANKINGS_PLANTILLA = {}
for nombre in VECTORES_PLANTILLA:
    informe, rankings = evaluar_plantilla(nombre)
    RANKINGS_PLANTILLA[nombre] = rankings
    BARRIDO_PLANTILLAS.append({
        "plantilla": nombre,
        **informe.summary,
        "chars_media": float(longitudes[nombre]),
        "pct_vs_A0": round(100 * float(longitudes[nombre]) / float(longitudes["A0"]), 1),
        "segundos": COSTES_PLANTILLA[nombre]["segundos"],
    })

barrido_plantillas = (
    pd.DataFrame(BARRIDO_PLANTILLAS).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
)
barrido_plantillas

,plantilla,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10,chars_media,pct_vs_A0,segundos
0,A3n,0.8375,0.4612,1.0,0.7945,158.9,12.1,42.00
1,A3,0.8125,0.4544,1.0,0.7757,151.0,11.5,45.58
2,A0,0.7875,0.4155,1.0,0.7718,1308.9,100.0,47.12
3,A1,0.7875,0.4424,1.0,0.7691,122.9,9.4,46.17
4,A4,0.8125,0.4334,1.0,0.7687,760.4,58.1,45.98
5,A2,0.8125,0.4137,1.0,0.7614,138.2,10.6,44.64
6,A5,0.7750,0.4009,1.0,0.7440,139.7,10.7,42.09


### C.1 · Las cuatro métricas, en una escala común

In [ ]:
METRICAS = ["precision_at_10", "recall_at_10", "mrr_at_10", "ndcg_at_10"]

plot_metric_comparison(
    {
        fila["plantilla"]: {m: float(fila[m]) for m in METRICAS}
        for _, fila in barrido_plantillas.iterrows()
    },
    title="Calidad por plantilla",
    subtitle=(
        f"{MODELO} [{CONTRATO}] @{DIM} · {CORPUS_ID} ({len(muestra)} docs) · "
        f"{len(query_ids)} consultas · k={TOP_K}"
    ),
).show()

### C.2 · Tabla por consulta

Una plantilla puede ganar de media y hundir dos consultas. Con 8 consultas, la media macro se mueve 0,125 por cada una que cambie de sitio, así que el agregado por sí solo no basta para decidir.

In [ ]:
por_consulta_plantilla = pd.concat([
    evaluar_plantilla(nombre)[0].per_query_frame().assign(plantilla=nombre)
    for nombre in VECTORES_PLANTILLA
])
por_consulta_plantilla.pivot(index="query_id", columns="plantilla", values="ndcg@10")

plantilla,A0,A1,A2,A3,A3n,A4,A5
query_id,,,,,,,
13357,0.6670,0.6337,0.7357,0.7825,0.7843,0.7955,0.7240
18868,0.4729,0.6556,0.7444,0.6607,0.6607,0.5059,0.5740
28703,0.9117,0.7722,0.7673,0.8348,0.9010,0.9094,0.8348
31224,0.7647,0.7461,0.8719,0.8353,0.8353,0.7647,0.7666
33633,0.6779,0.6865,0.3694,0.6171,0.6288,0.6575,0.5244
38249,0.8481,0.8151,0.8729,0.7458,0.8109,0.8521,0.7624
43240,0.9117,0.8437,0.7298,0.7837,0.7837,0.7269,0.8201
61533,0.9207,1.0000,1.0000,0.9458,0.9513,0.9371,0.9458


## D · A3 frente a A3n — el control de D02

Las dos plantillas se diferencian en **una sola cosa**: qué hacer cuando un campo está vacío. A3 omite la sección; A3n escribe `"Color: desconocido"`. Son unos pocos caracteres de diferencia de media — exactamente los 549 productos sin `color`.

**Cómo se lee la Δ:**

- **Δ > 0** (gana A3) → la hipótesis de contaminación se confirma: el literal compartido acerca productos que no se parecen. D02 era necesaria.
- **Δ ≈ 0** → D02 era una precaución sin coste. Se mantiene por prudencia, pero ya no como hecho medido.
- **Δ < 0** (gana A3n) → la decisión estaba equivocada. Rellenar aporta, y conviene revisar D02 antes de la ingesta final.

In [ ]:
if {"A3", "A3n"} <= set(VECTORES_PLANTILLA):
    a3 = barrido_plantillas.query("plantilla == 'A3'").iloc[0]
    a3n = barrido_plantillas.query("plantilla == 'A3n'").iloc[0]
    delta_d02 = round(float(a3["ndcg_at_10"]) - float(a3n["ndcg_at_10"]), 4)
    veredicto = (
        "gana A3 — la contaminación existe, D02 era necesaria" if delta_d02 > TOLERANCIA_R01
        else "gana A3n — rellenar aporta, hay que revisar D02" if delta_d02 < -TOLERANCIA_R01
        else "indistinguible — D02 era prudente, no imprescindible"
    )
    display(pd.DataFrame([{
        "A3 (omite)": a3["ndcg_at_10"],
        "A3n (rellena)": a3n["ndcg_at_10"],
        "delta": delta_d02,
        "veredicto": veredicto,
    }]))
    print(f"\nD02 · {veredicto}")
else:
    print("A3 o A3n sin codificar: el control de D02 no se puede evaluar.")

,A3 (omite),A3n (rellena),delta,veredicto
0,0.7757,0.7945,-0.0188,"gana A3n — rellenar aporta, hay que revisar D02"



D02 · gana A3n — rellenar aporta, hay que revisar D02


### D.1 · ¿Es señal o es perturbación?

Que la media suba no basta. Con 8 consultas, **una sola que cambie de sitio mueve la media macro 0,125** — más que de sobra para fabricar cualquier diferencia que veamos aquí. Antes de tocar D02 hay que responder a una pregunta distinta: *¿la mejora aparece donde el relleno actúa?*

El relleno solo toca a las fichas con `color` vacío. Así que cada consulta tiene una **exposición** medible: qué porcentaje de sus productos relevantes lleva ese campo en blanco.

**Y de ahí sale una predicción que se puede falsar.** Si rellenar aportara información sobre el color, las consultas más expuestas serían las que más se mueven — los puntos dibujarían una tendencia ascendente. Si en cambio el efecto aparece repartido al azar, y sobre todo si **la consulta con exposición total no se mueve**, entonces lo que estamos midiendo es que añadir texto compartido al corpus desplaza el espacio vectorial, no que aporte significado.

Es la diferencia entre un hallazgo y una casualidad con buena presencia.

In [ ]:
delta_d02 = per_query_delta(
    por_consulta_plantilla, sistema_a="A3n", sistema_b="A3", metrica="ndcg@10"
)
exposicion = relevant_field_nullity(relevancias, muestra, field="color")

control_d02 = (
    delta_d02
    .merge(exposicion, on="query_id")
    .merge(
        consultas.assign(query_id=consultas["query_id"].astype(str))[["query_id", "query_text"]],
        on="query_id",
    )
    .sort_values("pct_sin_color", ascending=False)
)

base_sin_color = round(100 * float(muestra["color"].isna().mean()), 1)
print(f"Linea base del catalogo: {base_sin_color}% de productos sin `color`")

control_d02[["query_id", "query_text", "A3", "A3n", "delta", "n_relevantes", "pct_sin_color"]]

Linea base del catalogo: 36.6% de productos sin `color`


,query_id,query_text,A3,A3n,delta,n_relevantes,pct_sin_color
3,61533,lentejas sin gluten,0.9458,0.9513,0.0055,30,100.0
0,28703,convertibles 2 en 1 portátil tactil,0.8348,0.9010,0.0662,39,25.6
2,33633,disfraz halloween talla grande hombre,0.6171,0.6288,0.0117,4,25.0
1,38249,estantes sin taladro habitacion,0.7458,0.8109,0.0651,35,22.9
4,13357,base tapizada 160x200 sin patas,0.7825,0.7843,0.0018,31,12.9
5,18868,botines marrones mujer tacon medio,0.6607,0.6607,0.0000,9,11.1
7,43240,funda ipad air 4 sin tapa,0.7837,0.7837,0.0000,35,5.7
6,31224,cámaras bridge baratas,0.8353,0.8353,0.0000,15,0.0


In [ ]:
plot_effect_vs_exposure(
    control_d02,
    exposure="pct_sin_color",
    effect="delta",
    tolerance=TOLERANCIA_R01,
    title="¿El relleno de nulos aporta información, o solo mueve el espacio?",
    subtitle=(
        "Un punto por consulta · eje X: % de sus productos relevantes con `color` vacío · "
        "eje Y: Δ nDCG@10 (A3n − A3) · la banda gris es la zona indistinguible"
    ),
).show()

### D.2 · Cómo se lee el gráfico

- **Tendencia ascendente** → a más exposición, más mejora: el relleno aporta información y **D02 estaba equivocada**.
- **Nube plana** → la mejora no tiene que ver con dónde actúa el relleno: es una perturbación del espacio vectorial, y **D02 se mantiene**.
- **El punto del extremo derecho es el que más pesa.** Es la consulta cuyos productos relevantes están *todos* sin color: la exposición máxima posible. Si esa no se mueve, ninguna hipótesis basada en el significado del relleno se sostiene.

Conviene además mirar el eje X contra la línea base del catálogo que imprime la celda anterior: una consulta por **debajo** de esa referencia está menos expuesta que el producto medio, así que una mejora grande ahí es todavía más difícil de explicar por el contenido del relleno.

## E · Consistencia sobre las 12 consultas ciegas

Las 8 consultas de desarrollo tienen juicios; las 12 de evaluación **no**, así que el nDCG es incalculable sobre ellas. Pero sí se puede medir algo que ninguna métrica con etiquetas captura: son **4 intenciones × 3 formulaciones** de lo mismo.

```
EVAL-100455-direct    "taladro 24v batería"
EVAL-100455-context   "taladro sin cable de 24 voltios que venga con su batería"
EVAL-100455-semantic  "quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe"
```

Un buscador semántico debería devolver **prácticamente los mismos productos** para las tres: piden lo mismo con palabras distintas. El Jaccard@10 entre formulaciones mide esa estabilidad.

**Por qué importa aquí:** el nDCG sobre 8 consultas se puede ganar por afinidad léxica con esas ocho concretas. La consistencia entre formulaciones dice qué plantilla **generaliza** a otra superficie léxica — y es la que se va a encontrar en producción, donde nadie escribe como el conjunto de desarrollo. Si una plantilla gana en nDCG pero pierde aquí, la ventaja probablemente era sobreajuste.

In [ ]:
consistencia = []
for nombre in VECTORES_PLANTILLA:
    docs = truncate_dim(VECTORES_PLANTILLA[nombre], DIM)
    retriever = DenseRetriever(docs, corpus_ids, metric=METRICA)
    rankings = rank_queries_dense(
        retriever, ciegas["evaluation_id"].tolist(), truncate_dim(vectores_ciegas, DIM), k=TOP_K
    )
    tabla = formulation_consistency(rankings, k=TOP_K)
    columnas = [c for c in tabla.columns if c.startswith("jaccard_")]
    consistencia.append({
        "plantilla": nombre,
        **{c: round(float(tabla[c].mean()), 4) for c in columnas},
        "jaccard_medio": round(float(tabla[columnas].to_numpy().mean()), 4),
    })

consistencia_plantillas = (
    pd.DataFrame(consistencia).sort_values("jaccard_medio", ascending=False).reset_index(drop=True)
)
consistencia_plantillas

,plantilla,jaccard_context_direct,jaccard_context_semantic,jaccard_direct_semantic,jaccard_medio
0,A2,0.5684,0.5989,0.4597,0.5423
1,A5,0.5617,0.5809,0.4471,0.5299
2,A3n,0.4904,0.5855,0.5030,0.5263
3,A1,0.5121,0.5571,0.4664,0.5119
4,A4,0.4987,0.5110,0.4151,0.4749
5,A3,0.5296,0.4561,0.4196,0.4685
6,A0,0.4792,0.4114,0.3750,0.4219


## F · R01 · Aplicar la regla y dejar el artefacto

**El criterio se fijó en `config.yaml` antes de ver la tabla**, igual que D09b en NB02:

```yaml
r01_criterio_desempate:
  forma: mas_corta_dentro_de_tolerancia
  metrica_primaria: ndcg_at_10
  tolerancia: 0.01
  coste: chars_media
```

1. `B` = mejor nDCG@10 de toda la tabla.
2. **Admisibles**: las que están a menos de 0,01 de `B`.
3. Entre las admisibles gana **la de menor longitud media**; a igualdad, mayor nDCG.

### ⚠️ A3n no entra en la elección

Su papel es el de **control de D02**, y estaba declarado antes de medir: no es otra receta de la secuencia, es la variante que existe para poner a prueba una decisión ya tomada. Se codifica y se mide igual que las demás —sin eso no habría con qué contrastar—, pero no aspira a ser la elegida.

Excluir a una plantilla **después** de ver que puntúa alto es exactamente lo que el enunciado penaliza, así que la declaración previa no basta por sí sola. Lo que sostiene la exclusión es el análisis de la sección D: su ventaja no aparece donde el relleno actúa, sino repartida al azar, y la consulta con exposición total al relleno es de las que menos se mueven. No se aparta porque incomode el resultado; se aparta porque se investigó de dónde venía y no venía de lo que la plantilla cambia.

**Por qué la longitud y no el número de campos.** El criterio buscado era *"que diga más con menos"*: densidad de significado. Pero "representar mejor el producto" es justo lo que mide nDCG@10, y el desempate **solo se activa cuando esa métrica ya ha declarado dos plantillas equivalentes**. En ese punto el significado está empatado por medición, así que maximizar densidad se reduce exactamente a minimizar el texto. Contar columnas habla de dependencia de datos, no de cuánto significado llevan.

> 🗳️ **R01 la ratificas tú** y la escribes en `config/config.yaml`. Si el resultado no te convence, el sitio para discutirlo es el criterio, no la tabla.

In [ ]:
# La regla se aplica solo a las CANDIDATAS. A3n queda fuera porque su papel es
# el de control de D02 —declarado en `aurum.plantillas.CONTROLES`, junto a la
# definición de la plantilla y antes de medir nada—, no el de aspirante.
#
# La exclusión no se sostiene sola: lo que la justifica es el análisis de la
# sección D, que mostró que su ventaja no viene de donde el relleno actúa.
candidatas_r01 = barrido_plantillas[~barrido_plantillas["plantilla"].isin(CONTROLES)]
print(f"candidatas: {sorted(candidatas_r01['plantilla'])}")
print(f"controles fuera de la regla: {sorted(CONTROLES)}")

ordenadas_plantillas = apply_tolerance_rule(
    candidatas_r01, metrica="ndcg_at_10", tolerancia=TOLERANCIA_R01,
    coste="chars_media", desempates=(),
)
ordenadas_plantillas[[
    "posicion_regla", "plantilla", "ndcg_at_10", "recall_at_10", "mrr_at_10",
    "chars_media", "pct_vs_A0", "admisible",
]]

candidatas: ['A0', 'A1', 'A2', 'A3', 'A4', 'A5']
controles fuera de la regla: ['A3n']


,posicion_regla,plantilla,ndcg_at_10,recall_at_10,mrr_at_10,chars_media,pct_vs_A0,admisible
0,1,A1,0.7691,0.4424,1.0,122.9,9.4,True
1,2,A3,0.7757,0.4544,1.0,151.0,11.5,True
2,3,A4,0.7687,0.4334,1.0,760.4,58.1,True
3,4,A0,0.7718,0.4155,1.0,1308.9,100.0,True
4,5,A2,0.7614,0.4137,1.0,138.2,10.6,False
5,6,A5,0.7440,0.4009,1.0,139.7,10.7,False


---

# G · El barrido sobre el catálogo completo

La sección F aplicó la regla sobre la muestra de desarrollo. Aquí se repite el barrido entero sobre los **15.000 productos**, que es el recorrido que evalúa el enunciado (§6), y **R01 se ratifica con estos números**, no con los de la muestra.

### Por qué no basta con la muestra

En NB02 el margen entre el modelo ganador y el baseline léxico era holgado, y aun así se estrechó un tercio al pasar de 1.500 a 15.000 candidatos. **Aquí los márgenes son de milésimas**: entre la plantilla más corta y la que venía de NB02 hay menos de 0,003 de nDCG@10. Una diferencia así no sobrevive a ninguna reordenación.

Y el mecanismo apunta en contra de las plantillas cortas. Con diez veces más productos compitiendo por las mismas diez posiciones, una representación de ~120 caracteres tiene mucho menos con lo que separar vecinos próximos que una de ~1.300. Cabría esperar que **el texto corto se degrade más** al añadir distractores — que es justo lo contrario de lo que necesita para ganar.

No es seguro, es la dirección en la que apunta el sentido común. Por eso se mide en vez de suponerse.

### ⚠️ Lo que cuesta

Siete plantillas × 15.000 documentos, unos **8 minutos cada una**: cerca de una hora. Y unos **1,3 GB** de vectores en la caché.

Es caro, pero es la decisión con la que se ingiere el catálogo definitivo en NB04: equivocarse obliga a recodificar los 15.000 y a reconstruir el índice desde cero. La celda estima el coste antes de lanzarlo y se niega a empezar si se dispara por encima del límite declarado.

> 📌 **El recorte de A4 cambia de valor, y es correcto.** Se deriva de la mediana del corpus que se codifica, así que sobre 15.000 no es el mismo número que sobre 1.500. A4 no es "la misma plantilla con otro corte": es la misma **regla** aplicada al corpus real.

In [ ]:
LIMITE_HORAS_TOTAL = 2.0      # por encima de esto la celda no lanza nada
CORPUS_COMPLETO = "catalogo_productos"

completo = pd.read_csv(DATA / "catalogo_productos.csv")
ids_completo = completo["product_id"].tolist()
CONTEXTO_COMPLETO = corpus_context(completo)

# Extrapolación desde el coste ya medido sobre la muestra: codificar es
# proporcional al número de documentos con el mismo modelo y el mismo lote.
seg_muestra = sum(c["segundos"] for c in COSTES_PLANTILLA.values())
horas = seg_muestra * len(completo) / len(muestra) / 3600

print(f"plantillas      : {len(TEMPLATES)}")
print(f"documentos      : {len(completo)} (x{len(completo) // len(muestra)} la muestra)")
print(f"corte A4        : {CONTEXTO_COMPLETO.a4_chars} chars (era {CONTEXTO.a4_chars} en la muestra)")
print(f"coste estimado  : ~{horas:.1f} h   ·   disco ~{len(TEMPLATES) * len(completo) * 3072 * 4 / 1e9:.1f} GB")

VECTORES_COMPLETO = {}
if horas > LIMITE_HORAS_TOTAL:
    print(f"\n⏭️  Por encima del límite de {LIMITE_HORAS_TOTAL} h: no se lanza.")
else:
    for nombre in TEMPLATES:
        inicio = time.perf_counter()
        textos = render_template(completo, nombre, context=CONTEXTO_COMPLETO)
        encoder = encoder_congelado()
        try:
            resultado = encode_corpus(
                encoder, textos, corpus_id=f"{CORPUS_COMPLETO}__{nombre}",
                kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
            )
        finally:
            del encoder
            gc.collect()
        VECTORES_COMPLETO[nombre] = resultado.vectors
        origen = "caché" if resultado.stats.desde_cache else f"{time.perf_counter() - inicio:.0f}s"
        print(f"  ✅ {nombre:<4} {resultado.vectors.shape} ({origen})")

plantillas      : 7
documentos      : 15000 (x10 la muestra)
corte A4        : 936 chars (era 1083 en la muestra)
coste estimado  : ~0.9 h   ·   disco ~1.3 GB
  ✅ A0   (15000, 3072) (501s)
  ✅ A1   (15000, 3072) (462s)
  ✅ A2   (15000, 3072) (486s)
  ✅ A3   (15000, 3072) (470s)
  ✅ A3n  (15000, 3072) (455s)
  ✅ A4   (15000, 3072) (472s)
  ✅ A5   (15000, 3072) (688s)


### G.1 · El barrido, a las dos escalas

La columna que decide es `ndcg_completo`. `ndcg_muestra` está al lado solo para ver **cuánto** se mueve cada plantilla al escalar: una caída homogénea no cambia nada, una caída desigual sí reordena.

In [ ]:
def evaluar_completo(nombre, *, dim=DIM, metric=METRICA):
    """Igual que `evaluar_plantilla`, pero contra los 15.000 IDs del catálogo.

    No se reutiliza aquella porque cerró sobre `corpus_ids`, que son los 1.500
    de la muestra: pasarle estos vectores devolvería identificadores
    equivocados sin lanzar ningún error."""
    docs = truncate_dim(VECTORES_COMPLETO[nombre], dim)
    queries = truncate_dim(vectores_query, dim)
    retriever = DenseRetriever(docs, ids_completo, metric=metric)
    rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
    return evaluate_rankings(rankings, qrels, k=TOP_K), rankings


if not VECTORES_COMPLETO:
    print("Sin codificar sobre el catálogo completo: la celda anterior no llegó a lanzarse.")
else:
    longitudes_completo = template_stats(completo).set_index("plantilla")["chars_media"]
    ndcg_muestra = barrido_plantillas.set_index("plantilla")["ndcg_at_10"]

    filas, RANKINGS_COMPLETO = [], {}
    for nombre in VECTORES_COMPLETO:
        informe, rankings = evaluar_completo(nombre)
        RANKINGS_COMPLETO[nombre] = rankings
        filas.append({
            "plantilla": nombre,
            **informe.summary,
            "chars_media": float(longitudes_completo[nombre]),
            "ndcg_muestra": float(ndcg_muestra[nombre]),
        })

    barrido_completo = (
        pd.DataFrame(filas).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
    )
    barrido_completo["caida_al_escalar"] = (
        barrido_completo["ndcg_at_10"] - barrido_completo["ndcg_muestra"]
    ).round(4)

    display(barrido_completo[[
        "plantilla", "ndcg_muestra", "ndcg_at_10", "caida_al_escalar",
        "recall_at_10", "chars_media",
    ]])

,plantilla,ndcg_muestra,ndcg_at_10,caida_al_escalar,recall_at_10,chars_media
0,A4,0.7687,0.6006,-0.1681,0.2905,654.3
1,A0,0.7718,0.5894,-0.1824,0.2632,1211.9
2,A1,0.7691,0.5681,-0.2010,0.2836,115.4
3,A3,0.7757,0.5668,-0.2089,0.2565,143.9
4,A3n,0.7945,0.5589,-0.2356,0.2253,152.2
5,A2,0.7614,0.5499,-0.2115,0.2430,131.2
6,A5,0.7440,0.5493,-0.1947,0.2119,132.1


### G.2 · R01 sobre el catálogo completo

La misma regla y la misma tolerancia declaradas de antemano, aplicadas ahora al corpus que de verdad se evalúa. Los controles siguen fuera del conjunto de candidatas.

**Esta tabla es la que ratifica R01.** Si la ganadora coincide con la de la muestra, la decisión llega respaldada a las dos escalas. Si no coincide, manda esta — y conviene dejar escrito en el informe que la muestra habría llevado a otra elección, porque es exactamente la trampa que el enunciado avisa en §6.

In [ ]:
if not VECTORES_COMPLETO:
    print("Sin barrido completo: R01 se queda con la ratificación sobre la muestra.")
else:
    candidatas_completo = barrido_completo[~barrido_completo["plantilla"].isin(CONTROLES)]
    ordenadas_completo = apply_tolerance_rule(
        candidatas_completo, metrica="ndcg_at_10", tolerancia=TOLERANCIA_R01,
        coste="chars_media", desempates=(),
    )

    g_muestra = ordenadas_plantillas.iloc[0]["plantilla"]
    g_completo = ordenadas_completo.iloc[0]["plantilla"]
    veredicto = (
        f"COINCIDEN: {g_completo} gana a las dos escalas"
        if g_muestra == g_completo
        else f"NO COINCIDEN: la muestra decía {g_muestra}, el catálogo completo dice {g_completo}"
    )
    print(veredicto)
    print(f"admisibles sobre el completo: {sorted(ordenadas_completo.query('admisible')['plantilla'])}")

    display(ordenadas_completo[[
        "posicion_regla", "plantilla", "ndcg_at_10", "recall_at_10",
        "mrr_at_10", "chars_media", "admisible",
    ]])

NO COINCIDEN: la muestra decía A1, el catálogo completo dice A4
admisibles sobre el completo: ['A4']


,posicion_regla,plantilla,ndcg_at_10,recall_at_10,mrr_at_10,chars_media,admisible
0,1,A4,0.6006,0.2905,0.9375,654.3,True
1,2,A1,0.5681,0.2836,0.9000,115.4,False
2,3,A2,0.5499,0.2430,0.8333,131.2,False
3,4,A5,0.5493,0.2119,0.8750,132.1,False
4,5,A3,0.5668,0.2565,0.8875,143.9,False
5,6,A0,0.5894,0.2632,0.8438,1211.9,False


### G.3 · Consistencia entre formulaciones, a escala real

La consistencia sobre las 12 consultas ciegas se midió antes con la muestra, y la decisión acabó tomándose sobre el catálogo completo. Dejarlas a escalas distintas invita a citar una comprobación que ya no acompaña a la elección, así que se repite aquí — y no cuesta nada: los vectores ya están codificados.

**Qué responde.** El nDCG se mide sobre 8 consultas concretas, y una plantilla puede ganarlas por afinidad léxica con ellas. El Jaccard entre las tres formulaciones de cada intención mide otra cosa: si el sistema devuelve **los mismos productos cuando le preguntan lo mismo con otras palabras**. Eso es lo que se encuentra en producción, donde nadie escribe como el conjunto de desarrollo.

**Cómo leerlo.** Si la ganadora de la regla también va bien aquí, la elección llega respaldada por dos medidas independientes —una con etiquetas y otra sin ellas—. Si va mal, hay un matiz que debe acompañar a la decisión en el informe.

> ⚠️ **Consistencia alta no es calidad.** Un sistema que devuelve los mismos diez productos equivocados para las tres formulaciones puntúa 1,0. Una plantilla con poca información puede ser muy estable justamente porque apenas responde a la consulta. Esta columna se lee **junto** al nDCG, nunca en su lugar.

In [ ]:
if not VECTORES_COMPLETO:
    print("Sin barrido completo: la consistencia se queda con la medida sobre la muestra.")
else:
    filas = []
    for nombre in VECTORES_COMPLETO:
        docs = truncate_dim(VECTORES_COMPLETO[nombre], DIM)
        retriever = DenseRetriever(docs, ids_completo, metric=METRICA)
        rankings = rank_queries_dense(
            retriever, ciegas["evaluation_id"].tolist(),
            truncate_dim(vectores_ciegas, DIM), k=TOP_K,
        )
        tabla = formulation_consistency(rankings, k=TOP_K)
        columnas = [c for c in tabla.columns if c.startswith("jaccard_")]
        filas.append({
            "plantilla": nombre,
            "jaccard_completo": round(float(tabla[columnas].to_numpy().mean()), 4),
        })

    ganadora_regla = ordenadas_completo.iloc[0]["plantilla"]
    consistencia_completo = (
        pd.DataFrame(filas)
        .merge(
            consistencia_plantillas[["plantilla", "jaccard_medio"]]
            .rename(columns={"jaccard_medio": "jaccard_muestra"}),
            on="plantilla",
        )
        .merge(barrido_completo[["plantilla", "ndcg_at_10"]], on="plantilla")
        .sort_values("jaccard_completo", ascending=False)
        .reset_index(drop=True)
    )
    consistencia_completo["gana_r01"] = consistencia_completo["plantilla"] == ganadora_regla

    puesto = int(consistencia_completo.index[
        consistencia_completo["plantilla"] == ganadora_regla
    ][0]) + 1
    print(f"{ganadora_regla} (ganadora de R01) queda {puesto}a de {len(consistencia_completo)} "
          f"en consistencia entre formulaciones")

    display(consistencia_completo[[
        "plantilla", "jaccard_muestra", "jaccard_completo", "ndcg_at_10", "gana_r01",
    ]])

A4 (ganadora de R01) queda 6a de 7 en consistencia entre formulaciones


,plantilla,jaccard_muestra,jaccard_completo,ndcg_at_10,gana_r01
0,A5,0.5299,0.5520,0.5493,False
1,A2,0.5423,0.5260,0.5499,False
2,A1,0.5119,0.5237,0.5681,False
3,A3n,0.5263,0.4965,0.5589,False
4,A3,0.4685,0.4543,0.5668,False
5,A4,0.4749,0.3747,0.6006,True
6,A0,0.4219,0.3727,0.5894,False


In [ ]:
def registros(frame):
    """Filas como tipos JSON nativos: `to_dict` dejaría escalares de numpy."""
    return json.loads(frame.to_json(orient="records"))


# R01 se ratifica sobre el CATALOGO COMPLETO, no sobre la muestra: el orden
# entre plantillas cambia al escalar, y decidir con 1.500 candidatos habria
# llevado a otra plantilla. Si la seccion G no llego a lanzarse se cae a la
# muestra, pero queda dicho en el artefacto que no esta confirmada.
if VECTORES_COMPLETO:
    ganadora = ordenadas_completo.iloc[0]
    decidida_sobre = CORPUS_COMPLETO
else:
    ganadora = ordenadas_plantillas.iloc[0]
    decidida_sobre = f"{CORPUS_ID} (SIN confirmar a escala real)"

artefacto = {
    "configuracion": {
        "corpus": CORPUS_ID,
        "n_docs": len(muestra),
        "modelo_congelado": {
            "id": MODELO, "contrato": CONTRATO, "dim": DIM, "metrica": METRICA,
        },
        "top_k": TOP_K,
        "r01": {
            "metrica": "ndcg_at_10", "tolerancia": TOLERANCIA_R01,
            "coste": "chars_media", "decidida_sobre": decidida_sobre,
            "ganadora": ganadora["plantilla"],
        },
        "d07_chunking": False,
    },
    "plantillas": registros(template_stats(muestra)),
    "errores_de_codificacion": ERRORES_PLANTILLA,
    "costes_de_codificacion": list(COSTES_PLANTILLA.values()),
    "barrido": registros(barrido_plantillas),
    "regla_r01": registros(ordenadas_plantillas),
    "consistencia_ciegas": registros(consistencia_plantillas),
    # El barrido a escala real, que es el que ratifica R01. Puede no existir si
    # la seccion G no llego a lanzarse.
    "barrido_completo": registros(barrido_completo) if VECTORES_COMPLETO else [],
    "regla_r01_completo": registros(ordenadas_completo) if VECTORES_COMPLETO else [],
    "consistencia_ciegas_completo": (
        registros(consistencia_completo) if VECTORES_COMPLETO else []
    ),
    "rankings": RANKINGS_PLANTILLA,
}

destino = Path("..") / "artifacts" / "comparativa_representacion.json"
destino.write_text(json.dumps(artefacto, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

markdown = Path("..") / "artifacts" / "comparativa_representacion.md"
markdown.write_text(
    "# Comparativa de representacion (NB03)\n\n"
    f"Modelo congelado: `{MODELO}` [{CONTRATO}] @{DIM} · {METRICA}\n"
    f"Corpus: `{CORPUS_ID}` ({len(muestra)} docs) · k={TOP_K}\n\n"
    "## Barrido de plantillas\n\n" + barrido_plantillas.to_markdown(index=False) + "\n\n"
    "## Regla R01 aplicada\n\n" + ordenadas_plantillas.to_markdown(index=False) + "\n\n"
    "## Consistencia entre formulaciones (12 consultas ciegas)\n\n"
    + consistencia_plantillas.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
print(f"R01 ratificada sobre {decidida_sobre}")
print(f"  ganadora: {ganadora['plantilla']} "
      f"(nDCG@10 = {ganadora['ndcg_at_10']}, {ganadora['chars_media']:.0f} chars de media)")
print(f"Escrito {destino.name} y {markdown.name}")

R01 ratificada sobre catalogo_productos
  ganadora: A4 (nDCG@10 = 0.6006, 654 chars de media)
Escrito comparativa_representacion.json y comparativa_representacion.md
